In [ ]:
import os
# Disable GPU usage (GPUs add no value for the small example and we do not need to fight for ressources this way) 
os.environ["CUDA_VISIBLE_DEVICES"]="-1"
import tensorflow as tf

# Gesten erkennen mit Neuronalen Netzwerken

Wir wollen noch einmal den Gestendatensatz aus Übung Nummer 6 verwenden. Diesmal wollen wir diesen aber mit einem neuronalen Netzwerk klassifizieren.

Zuerst müssen wir die Daten laden. Die Tensorflow Methode [image_dataset_from_directory](https://www.tensorflow.org/api_docs/python/tf/keras/utils/image_dataset_from_directory) erledigt das für uns, unter der Annahme, dass die Bilder in eigenen Verzeichnissen je Klasse liegen (was bei uns der Fall ist). Nötige Parameter:
- `directory`: Das Verzeichnis, in dem sich die Unterverzeichnisse je Klasse befinden
- `shuffle`: Wollen wir die Bilder zufällig permutieren (Ja, ansonsten kommen ja z.B. erst alle 0en, dann alle 1en....)
- `batch_size`: Wir trainieren mit minibatch stochastic gradient descent (Mittelweg zwischen nur einem Beispiel je Schritt und allen Beispielen). Hier geben wir die Größe eines Minibatches an (64 passst ganz gut).
- `image_size`: Ein Tupel mit der Zielgröße, in die die Bilder skaliert werden sollen. Wir wollen 40x40 nehmen.
- `validation_split`: Welchen Anteil wollen wir als Validation-Set behalten (20%)
- `subset`: Entweder "training" oder "validation". Gibt an, welches der beiden Teil Sets zurückgegeben werden soll.
- `seed`: Random Seed für Shuffle. Muss der gleiche für das Trainings- und Validation-Set sein, damit diese sicher disjunkt sind.

Lesen Sie nun ein Trainings- und ein Validation-Set ein.

In [ ]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

train_dataset = # ADD CODE HERE
validation_dataset = # ADD CODE HERE

Ein Dataset verhält sich ungefähr wie ein Generator. Man kann darüber iterieren (ohne dass alle Daten notwendigerweise im Speicher liegen müssen).
Wir können mit `element_spec` schauen, welche Form unsere Daten haben.

In [ ]:
train_dataset.element_spec

In [ ]:
for x in train_dataset:
    print(x)
    break

## Logistische Regression

Als erstes wollen wir einmal logistische Regression mit Tensorflow machen.
Mit `tf.keras.models.Sequential` können wir ein Modell erstellen, das aus mehreren Layern besteht (das können wir dann später einfach um mehr ebenen Erweitern). Diesem geben wir eine Liste mit Layern mit.

- `tf.keras.layers.Rescaling`: Ist ein Layer, um unsere Input-Features zu skalieren (wieder sollten wir durch 255 teilen, damit die Werte zwischen 0 und 1 liegen. Probieren Sie mal aus, was passiert, wenn wir das weglassen.
- `tf.keras.layers.Flatten`: Ist ein Layer, der die 40x40x3 Bilder in einen langen Vektor überführt.
- `tf.keras.layers.Dense` Ist ein regulärer linearer Layer, so wie wir ihn kennen. Diesem geben wir mit
    - `units`: Wieviele Outputs soll der Layer haben (wir haben 10 Klassen, also brauchen wir auch so viele Outputs)
    - `activation`: Welche Aktivierungsfunktion soll verwendet werden (für unseren Fall ist "softmax" die richtige)
    - `kernel_regularizer`: Wie soll regularisiert werden. Hier können wir L2 (`tf.keras.regularizers.L2`) Regularisierung mit einem Gewicht von 0.01 nehmen (Sie können gerne etwas experimentieren, was hier gut funktioniert.)
    
Mit dem definierten Modell müssen wir nun spezifizieren, wie es optimiert werden soll. Dafür rufen wir `model.compile` auf. Diesem müssen wir mitgeben:
- `optimizer`: Welcher Optimierungsalgorithmus verwendet werden soll. `tf.optimizers.SGD` ist normaler Stochastic Gradient Descent (auf Minibatches von der Größe, wie sie vom Datensatz geliefert werden). Wir müssen hier eine Lernrate angeben. Probieren Sie aus, was gut funktioniert.
- `loss`: Welche Zielfunktion soll optimiert werden. Für Klassifikation ist "CategoricalCrossentropy" Oft die richtige Wahl. Hier (und in vielen anderen Fällen) liegen die Klassen als Integer (0, 1, 2, ..., 9) vor. Mit der Loss Funktion `tf.losses.SparseCategoricalCrossentropy` brauchen wir nicht selber ein One-Hot Encoding der Klassen zu machen, sondern Tensorflow weiß, dass es das intern machen soll.

Ein weiterer Parameter, den wir angeben können, ist:
- `metrics`: Eine Liste mit zusätzlichen Metriken, die getrackt werden sollen. Hier können wir `tf.metrics.SparseCategoricalAccuracy` angeben, um die Accuracy direkt mit ausgedruckt zu bekommen.

Als letztes müssen wir das Modell trainieren. Das tun wir mit `model.fit`. Hier geben wir mit:
- Das train_dataset
- `epochs`: Die Anzahl an vollständigen Durchläufen durch das gesamte Trainingsset. Mit circa 200-300 kann man zu brauchbaren Ergebnissen kommen.

Optional können wir direkt noch mit angeben:
- `validation_data`: Wenn wir ein Validation Set mit angeben, dann bekommen wir nach jeder Epoche auch alle Metriken für die Validation Daten berechnet.

In [ ]:
model = tf.keras.models.Sequential([
    # ADD CODE HERE
])

# ADD CODE HERE

## Neuronales Netzwerk

Ein neuronales Netzwerk ist jetzt leicht definiert. Wir fügen einfach mehr Layer hinzu. Fügen Sie zum Beispiel nach dem "Flatten" Layer noch zwei Layer mit 50 bzw. 25 Outputs und "relu" Activations hinzu.

Hier sieht man schnell, was Neuronale Netze schwierig in der Handhabung macht. Auf einmal haben wir eine sehr große Anzahl an Hyperparametern, die eingestellt werden müssen.

Beim Versuch das Netzwerk zu trainieren, fällt Ihnen eventuell auf, dass das Training am Anfang guten Fortschritt macht, während dann später der Loss immer öfter Sprünge nach oben macht. Je besser wir werden, umso mehr macht eine kleine Lernrate Sinn, damit wir den Fortschritt nicht mehr verlieren.
Mit `tf.keras.optimizers.schedules.ExponentialDecay` können wir eine sinkende Lernrate implementieren, die wir dann "SGD" als Parameter mitgeben. Hier können wir angeben:
- `initial_learning_rate`: Die Lernrate im ersten Schritt (im ersten Mini Batch)
- `decay_steps`: Nach wievielen Minibatches soll die Lernrate jeweils verringert werden
- `decay_rate`: Um welchen Faktor soll die Lernrate jeweils verringert werden (typischerweise >= 0.9 und < 1.0).

Mit 3 Layern kommen wir auf ein Ergebnis, dass über dem aus der Cross Validation-Übung liegt. Schaffen Sie es, auch besser als die SVM zu werden? Ich jedenfalls nicht ;-)

In [ ]:
# ADD CODE HERE

Mit `model.summary()` können Sie sich eine Zusammenfassung des Modells drucken. Unter anderem können wir hier direkt sehen, dass unser Modell fast eine Viertelmillion Parameter hat.

In [ ]:
model.summary()

Sie müssen mit dem Training eines Modells nicht bei 0 anfangen. Indem Sie erneut `.fit` aufrufen, können Sie das Training eines Modell einfach mit ein paar zusätzlichen Epochen fortsetzen. Dabei können Sie `initial_epoch` angeben, damit die Funktion weiß, wo sie aufgehört hat (wichtig für das Scaling der Lernrate). Die Anzahl Epochen vom `epoch` Parameter ist dabei nicht die Anzahl an Extra Epochen, sondern wieviele Sie ingesamt trainieren möchten.